# SNIES 2020–2025 — Normalización, EDA y dimensiones de programas

**Objetivo.** Construir una base de trabajo reproducible para el proyecto de análisis de mercado sin incorporar todavía CECOS. El flujo parte del histórico SNIES, normaliza los cambios de estructura entre años, realiza un EDA, extrae los programas de la Ibero (código IES **2830**), genera una primera homologación automática de nombres de programas, prepara una base para homologación manual y construye la dimensión de universidades.

### Salidas esperadas
1. `01_Ibero_2830_programas.xlsx`
2. `02_Dim_Programa_Agrupacion_Automatica.xlsx`
3. `03_Programas_Normalizados_Manual.xlsx`
4. `04_Dim_Universidades.xlsx`
5. `05_EDA_SNIES.xlsx`

> La homologación automática es una **primera propuesta**. Los matches de confianza media/baja deben revisarse manualmente antes de usarlos como regla productiva.

## 1. Configuración del proyecto

La estructura esperada es:

```text
03_Data_Preparation
├─ notebook/
│  └─ 01_SNIES_Normalizacion_EDA_Dimensiones.ipynb
└─ data/
   ├─ raw/
   │  └─ Historico_SNIES.xlsx
   └─ processed/
```

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import csv
import gzip
import html
import json
import re
import unicodedata
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

# Salidas y visualización
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import matplotlib.pyplot as plt


# CONFIGURACIÓN DEL PROYECTO

CWD = Path.cwd().resolve()

# Busca hacia arriba la raíz que contiene data/raw
PROJECT = next(
    (
        carpeta
        for carpeta in [CWD, *CWD.parents]
        if (carpeta / "data" / "raw").exists()
    ),
    None
)

if PROJECT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta raíz del proyecto con data/raw."
    )

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

INPUT = RAW / "Historico_SNIES.xlsx"

CODIGO_IBERO = "2830"

# Solo activar si queremos exportar el histórico normalizado completo.
EXPORTAR_HISTORICO_NORMALIZADO = False


# ============================================================
# VALIDACIÓN
# ============================================================

assert INPUT.exists(), f"No se encontró: {INPUT.resolve()}"

print("Raíz del proyecto:", PROJECT)
print("Archivo fuente:", INPUT)
print("Carpeta de salidas:", PROCESSED)


Raíz del proyecto: C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics
Archivo fuente: C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\raw\Historico_SNIES.xlsx
Carpeta de salidas: C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\processed


## 2. Carga y homologación estructural del histórico SNIES

Se cargan automáticamente todas las hojas cuyo nombre corresponde a un año.

El esquema de 2025 se utiliza como referencia canónica para homologar
las columnas históricas. Las nuevas columnas incorporadas en años
posteriores se conservan y se agregan con valor 0 a los años en los
que estructuralmente no existían.

AÑO_ORIGEN se conserva como campo de trazabilidad.

In [3]:
# 2.1 DETECCIÓN AUTOMÁTICA DE AÑOS

excel_snies = pd.ExcelFile(INPUT)

ANIOS = sorted(
    hoja
    for hoja in excel_snies.sheet_names
    if re.fullmatch(r"\d{4}", str(hoja))
)

print("Años detectados:", ANIOS)

Años detectados: ['2020', '2021', '2022', '2023', '2024', '2025']


In [4]:
# 2.2 ESQUEMA CANÓNICO 2025

COLUMNAS_2025 = list(
    pd.read_excel(
        INPUT,
        sheet_name="2025",
        nrows=0
    ).columns
)

# Columna residual eliminada también en Power Query
COLUMNAS_2025 = [
    columna
    for columna in COLUMNAS_2025
    if columna != "Column42"
]

print("Columnas esperadas según 2025:", len(COLUMNAS_2025))

Columnas esperadas según 2025: 41


In [5]:
# 2.3 COLUMNAS RENOMBRADAS MANUALES

RENOMBRES_COLUMNAS = {
    "IES_PADRE": "IES PADRE",
    "PRINCIPAL O SECCIONAL": "TIPO IES",
    "ID CARACTER": "ID CARÁCTER IES",
    "CARACTER IES": "CARÁCTER IES",
    "CÓDIGO DEL MUNICIPIO (IES)": "CÓDIGO DEL MUNICIPIO IES",
    "ID METODOLOGÍA": "ID MODALIDAD",
    "METODOLOGÍA": "MODALIDAD",
    "ID ÁREA DE CONOCIMIENTO": "ID ÁREA",
    "ID CINE CODIGO DETALLADO": "ID CINE CAMPO DETALLADO",
    "DESC CINE CODIGO DETALLADO": "DESC CINE CAMPO DETALLADO",
    "PRIMER CURSO": "MATRICULADOS PRIMER CURSO",
}

## Se lee únicamente los encabezados de una hoja y aplica los mismos renombres definidos en Power Query

def obtener_columnas_homologadas(anio):

    columnas = list(
        pd.read_excel(
            INPUT,
            sheet_name=anio,
            nrows=0
        ).columns
    )

    # Equivalente a MissingField.Ignore
    columnas = [
        RENOMBRES_COLUMNAS.get(columna, columna)
        for columna in columnas
    ]

    # Column42 es residual y en Power Query se elimina
    columnas = [
        columna
        for columna in columnas
        if columna != "Column42"
    ]

    return columnas

In [6]:
# 2.4 DIAGNOSTICO

validacion_columnas = []

for anio in ANIOS:

    columnas_anio = obtener_columnas_homologadas(anio)

    faltantes = [
        columna
        for columna in COLUMNAS_2025
        if columna not in columnas_anio
    ]

    adicionales = [
        columna
        for columna in columnas_anio
        if columna not in COLUMNAS_2025
    ]

    validacion_columnas.append({
        "AÑO": anio,
        "COLUMNAS_AÑO": len(columnas_anio),
        "COLUMNAS_2025": len(COLUMNAS_2025),
        "FALTANTES": len(faltantes),
        "ADICIONALES": len(adicionales),
        "DETALLE_FALTANTES": faltantes,
        "DETALLE_ADICIONALES": adicionales,
    })

df_validacion_columnas = pd.DataFrame(validacion_columnas)

df_validacion_columnas

,AÑO,COLUMNAS_AÑO,COLUMNAS_2025,FALTANTES,ADICIONALES,DETALLE_FALTANTES,DETALLE_ADICIONALES
0,2020,39,41,2,0,"[IES ACREDITADA, PROGRAMA ACREDITADO]",[]
1,2021,41,41,0,0,[],[]
2,2022,41,41,0,0,[],[]
3,2023,41,41,0,0,[],[]
4,2024,41,41,0,0,[],[]
5,2025,41,41,0,0,[],[]


In [7]:
# 2.5 VALIDACIÓN DE PARES 

def validar_renombres(anio):
    """
    Muestra qué columnas originales fueron renombradas
    y cuál es su nombre final.
    """

    columnas_originales = list(
        pd.read_excel(
            INPUT,
            sheet_name=anio,
            nrows=0
        ).columns
    )

    registros = []

    for columna in columnas_originales:

        columna_final = RENOMBRES_COLUMNAS.get(
            columna,
            columna
        )

        registros.append({
            "AÑO": anio,
            "COLUMNA_ORIGINAL": columna,
            "COLUMNA_FINAL": columna_final,
            "RENOMBRADA": columna != columna_final,
            "EXISTE_EN_2025": columna_final in COLUMNAS_2025,
        })

    return pd.DataFrame(registros)

df_mapeo_columnas = pd.concat(
    [
        validar_renombres(anio)
        for anio in ANIOS
    ],
    ignore_index=True
)

df_mapeo_columnas[
    df_mapeo_columnas["RENOMBRADA"]
]

,AÑO,COLUMNA_ORIGINAL,COLUMNA_FINAL,RENOMBRADA,EXISTE_EN_2025
1,2020,IES_PADRE,IES PADRE,True,True
3,2020,PRINCIPAL O SECCIONAL,TIPO IES,True,True
6,2020,ID CARACTER,ID CARÁCTER IES,True,True
7,2020,CARACTER IES,CARÁCTER IES,True,True
10,2020,CÓDIGO DEL MUNICIPIO (IES),CÓDIGO DEL MUNICIPIO IES,True,True
18,2020,ID METODOLOGÍA,ID MODALIDAD,True,True
19,2020,METODOLOGÍA,MODALIDAD,True,True
20,2020,ID ÁREA DE CONOCIMIENTO,ID ÁREA,True,True
28,2020,ID CINE CODIGO DETALLADO,ID CINE CAMPO DETALLADO,True,True
29,2020,DESC CINE CODIGO DETALLADO,DESC CINE CAMPO DETALLADO,True,True


In [8]:
# 2.6 CONSOLIDADO AÑO

def cargar_anio(anio):
    """
    Carga una hoja SNIES y replica las transformaciones
    estructurales realizadas en Power Query.
    """

    df = pd.read_excel(
        INPUT,
        sheet_name=anio
    )

    # 1. Homologar nombres de columnas
    df = df.rename(
        columns=RENOMBRES_COLUMNAS
    )

    # 2. Eliminar columna residual si existe
    df = df.drop(
        columns=["Column42"],
        errors="ignore"
    )

    # 3. Caso específico de 2020:
    #    estas columnas no existían en la fuente original
    if anio == "2020":

        if "IES ACREDITADA" not in df.columns:
            df["IES ACREDITADA"] = pd.NA

        if "PROGRAMA ACREDITADO" not in df.columns:
            df["PROGRAMA ACREDITADO"] = pd.NA

    # 4. Campo de trazabilidad
    df["AÑO_ORIGEN"] = int(anio)

    return df

historicos = {
    anio: cargar_anio(anio)
    for anio in ANIOS
}

In [9]:
# 2.7 VALIDACIÓN DE ESTRUCTURA POR AÑO

validacion_estructura = []

for anio, df in historicos.items():

    columnas_anio = [
        columna
        for columna in df.columns
        if columna != "AÑO_ORIGEN"
    ]

    faltantes = [
        columna
        for columna in COLUMNAS_2025
        if columna not in columnas_anio
    ]

    adicionales = [
        columna
        for columna in columnas_anio
        if columna not in COLUMNAS_2025
    ]

    validacion_estructura.append({
        "AÑO": anio,
        "COLUMNAS_AÑO": len(columnas_anio),
        "COLUMNAS_2025": len(COLUMNAS_2025),
        "FALTANTES": len(faltantes),
        "ADICIONALES": len(adicionales),
        "DETALLE_FALTANTES": faltantes,
        "DETALLE_ADICIONALES": adicionales,
    })

df_validacion_estructura = pd.DataFrame(
    validacion_estructura
)

df_validacion_estructura

,AÑO,COLUMNAS_AÑO,COLUMNAS_2025,FALTANTES,ADICIONALES,DETALLE_FALTANTES,DETALLE_ADICIONALES
0,2020,41,41,0,0,[],[]
1,2021,41,41,0,0,[],[]
2,2022,41,41,0,0,[],[]
3,2023,41,41,0,0,[],[]
4,2024,41,41,0,0,[],[]
5,2025,41,41,0,0,[],[]


In [10]:

# 2.8 CONSOLIDACIÓN DEL HISTÓRICO

df_historico = pd.concat(
    [
        historicos[anio]
        for anio in ANIOS
    ],
    ignore_index=True
)

print(
    f"Histórico consolidado: "
    f"{len(df_historico):,} filas x "
    f"{len(df_historico.columns)} columnas"
)

Histórico consolidado: 308,579 filas x 42 columnas


In [11]:
df_historico.tail()

,CÓDIGO DE LA INSTITUCIÓN,IES PADRE,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),TIPO IES,ID SECTOR IES,SECTOR IES,ID CARÁCTER IES,CARÁCTER IES,CÓDIGO DEL DEPARTAMENTO (IES),DEPARTAMENTO DE DOMICILIO DE LA IES,...,CÓDIGO DEL MUNICIPIO (PROGRAMA),MUNICIPIO DE OFERTA DEL PROGRAMA,ID SEXO,SEXO,AÑO,SEMESTRE,MATRICULADOS PRIMER CURSO,IES ACREDITADA,PROGRAMA ACREDITADO,AÑO_ORIGEN
308574,La información suministrada corresponde a lo r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308575,"A partir de 2021, la Universidad Internacional...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308576,"Para el año 2025, de las 305 IES con personerí...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308577,A partir de la información de la vigencia 2019...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025
308578,La información de acreditación de IES y progra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025


In [12]:
# 2.9 REGISTROS VÁLIDOS PARA LOS COMENTARIOS QUE DESCARGA EL REPORTE
filas_antes = len(df_historico)

codigo_ies_numerico = pd.to_numeric(
    df_historico["CÓDIGO DE LA INSTITUCIÓN"],
    errors="coerce"
)

df_historico = df_historico[
    codigo_ies_numerico.notna()
].copy()

filas_despues = len(df_historico)

print(f"Registros antes del filtro: {filas_antes:,}")
print(f"Registros válidos:          {filas_despues:,}")
print(f"Registros eliminados:       {filas_antes - filas_despues:,}")

Registros antes del filtro: 308,579
Registros válidos:          308,555
Registros eliminados:       24


In [13]:
df_historico.tail()

,CÓDIGO DE LA INSTITUCIÓN,IES PADRE,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),TIPO IES,ID SECTOR IES,SECTOR IES,ID CARÁCTER IES,CARÁCTER IES,CÓDIGO DEL DEPARTAMENTO (IES),DEPARTAMENTO DE DOMICILIO DE LA IES,...,CÓDIGO DEL MUNICIPIO (PROGRAMA),MUNICIPIO DE OFERTA DEL PROGRAMA,ID SEXO,SEXO,AÑO,SEMESTRE,MATRICULADOS PRIMER CURSO,IES ACREDITADA,PROGRAMA ACREDITADO,AÑO_ORIGEN
308566,9941,9941.0,CORPORACIÓN INSTITUCIÓN UNIVERSITARIA SALUD CO...,Principal,2.0,Privado,3.0,Institución Universitaria/Escuela Tecnológica,73.0,Tolima,...,73001.0,Ibagué,2.0,Femenino,2025.0,2.0,20.0,No,No,2025
308567,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,1.0,Masculino,2025.0,2.0,11.0,Si,No,2025
308568,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,2.0,Femenino,2025.0,2.0,9.0,Si,No,2025
308569,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,1.0,Masculino,2025.0,2.0,7.0,Si,No,2025
308570,9943,1704.0,UNIVERSIDAD SANTO TOMAS,Seccional,2.0,Privado,4.0,Universidad,50.0,Meta,...,50001.0,Villavicencio,2.0,Femenino,2025.0,2.0,3.0,Si,No,2025


## 3. Construcción de la dimensión de universidades

A partir del histórico SNIES consolidado (`df_historico`), se construye una dimensión única de instituciones de educación superior para ser utilizada posteriormente en el modelo de Power BI.

La dimensión tendrá como llave principal el código de institución SNIES (`ID_UNIVERSIDAD`) y contendrá una única fila por universidad.

El proceso incluye:

- normalización del código institucional;
- limpieza del nombre de la institución;
- revisión de variaciones históricas del nombre;
- selección del nombre institucional más reciente disponible;
- validación de unicidad de la llave;
- exportación del archivo `04_Dim_Universidades.xlsx`.

El objetivo es obtener una dimensión estable y reutilizable que permita relacionar el histórico SNIES con los atributos institucionales en el modelo analítico.

In [14]:
# 3.1 FUNCIONES DE NORMALIZACIÓN

def normalizar_codigo(serie):
    """
    Convierte códigos numéricos almacenados como texto o decimal
    a texto entero limpio.
    
    """

    numerico = pd.to_numeric(
        serie,
        errors="coerce"
    )

    return (
        numerico
        .astype("Int64")
        .astype("string")
    )


def normalizar_texto(serie):
    """
    Limpieza básica de campos de texto sin modificar
    su contenido semántico.

    - elimina BOM
    - normaliza Unicode
    - elimina espacios repetidos
    - elimina espacios iniciales/finales
    """

    return (
        serie
        .astype("string")
        .str.replace("\ufeff", "", regex=False)
        .str.replace("<U+FEFF>", "", regex=False)
        .str.replace("\xa0", " ", regex=False)
        .str.normalize("NFKC")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

In [15]:
# 3.2 NORMALIZACIÓN DEL CÓDIGO DE UNIVERSIDAD

df_historico["CÓDIGO DE LA INSTITUCIÓN"] = normalizar_codigo(
    df_historico["CÓDIGO DE LA INSTITUCIÓN"]
)

# validación 

print(
    "Universidades únicas:",
    df_historico["CÓDIGO DE LA INSTITUCIÓN"].nunique()
)

print(
    "Códigos nulos:",
    df_historico["CÓDIGO DE LA INSTITUCIÓN"].isna().sum()
)

print(
    "Ibero 2830 encontrada:",
    CODIGO_IBERO in set(
        df_historico["CÓDIGO DE LA INSTITUCIÓN"].dropna()
    )
)

Universidades únicas: 336
Códigos nulos: 0
Ibero 2830 encontrada: True


In [16]:
# 3.3 BASE PARA DIMENSIÓN DE UNIVERSIDADES

df_universidades_base = df_historico[
    [
        "CÓDIGO DE LA INSTITUCIÓN",
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        "AÑO_ORIGEN",
    ]
].copy()

# Normalización del nombre de la institución
df_universidades_base[
    "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
] = normalizar_texto(
    df_universidades_base[
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
    ]
)

#VALIDACIÓN
variaciones_nombre = (
    df_universidades_base
    .groupby(
        "CÓDIGO DE LA INSTITUCIÓN"
    )["INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Instituciones con más de un nombre:",
    (variaciones_nombre > 1).sum()
)

variaciones_nombre.head(20)

Instituciones con más de un nombre: 49


CÓDIGO DE LA INSTITUCIÓN
3828    3
9903    3
2743    3
9131    2
1206    2
4817    2
4811    2
4837    2
9102    2
4111    2
4110    2
4107    2
4106    2
3821    2
3808    2
4112    2
4835    2
3703    2
3725    2
1215    2
Name: INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES), dtype: int64

In [17]:
# 3.4  DETALLE DE INSTITUCIONES CON VARIACIONES

codigos_con_variaciones = variaciones_nombre[
    variaciones_nombre > 1
].index

df_variaciones_universidad = (
    df_universidades_base[
        df_universidades_base[
            "CÓDIGO DE LA INSTITUCIÓN"
        ].isin(codigos_con_variaciones)
    ]
    [
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        ]
    )
    .reset_index(drop=True)
)

df_variaciones_universidad.head(10)

,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)
0,1206,UNIVERSIDAD DE NARI¿O
1,1206,UNIVERSIDAD DE NARIÑO
2,1214,UNIVERSIDAD DE CUNDINAMARCA
3,1214,UNIVERSIDAD DE CUNDINAMARCA-UDEC
4,1215,UNIVERSIDAD DE CUNDINAMARCA
5,1215,UNIVERSIDAD DE CUNDINAMARCA-UDEC
6,1216,UNIVERSIDAD DE CUNDINAMARCA
7,1216,UNIVERSIDAD DE CUNDINAMARCA-UDEC
8,1833,CORPORACIÓN UNIVERSIDAD DEL SINU - ELIAS BECHA...
9,1833,UNIVERSIDAD DEL SINU - ELIAS BECHARA ZAINUM - ...


In [18]:
# 3.5 VALIDACIÓN DE CARACTERES SOSPECHOSOS

df_nombres_sospechosos = (
    df_universidades_base[
        df_universidades_base[
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
        ].str.contains(
            r"[¿�]",
            regex=True,
            na=False
        )
    ]
    [
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
            "AÑO_ORIGEN",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "AÑO_ORIGEN",
        ]
    )
    .reset_index(drop=True)
)

df_nombres_sospechosos

,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),AÑO_ORIGEN
0,1206,UNIVERSIDAD DE NARI¿O,2020
1,2707,FUNDACI¿N UNIVERSITARIA JUAN N. CORPAS,2020
2,2719,UNIVERSIDAD CAT¿LICA LUIS AMIG¿,2020
3,2731,FUNDACION UNIVERSITARIA CATOLICA LUMEN GENTIUM...,2020
4,2745,FUNDACI¿N UNIVERSITARIA COMPENSAR,2020
5,3828,CORPORACI¿N TECNOL¿GICA DE EDUCACI¿N SUPERIOR ...,2020
6,4835,CORPORACION UNIVERSITARIA TALLER CINCO CENTRO ...,2020
7,9131,FUNDACI¿N UNIVERSITARIA CERVANTES SAN AGUST¿N ...,2020
8,9914,"ESEIT ¿ ESCUELA SUPERIOR DE EMPRESA, INGENIER¿...",2020
9,9929,Universidad Aut¿noma Ind¿gena Intercultural - ...,2020


In [19]:
# 3.6 SELECCIÓN DE LA DENOMINACIÓN DE LA IES

# Identificar nombres con posibles problemas de codificación
df_universidades_base["NOMBRE_SOSPECHOSO"] = (
    df_universidades_base[
        "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)"
    ]
    .str.contains(
        r"[¿�ÃÂ]",
        regex=True,
        na=False
    )
)

# CANDIDATOS DE NOMBRE POR INSTITUCIÓN Y AÑO

df_candidatos_ies = (
    df_universidades_base
    .groupby(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
            "AÑO_ORIGEN",
            "NOMBRE_SOSPECHOSO",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="FRECUENCIA")
)


# CANDIDATOS VÁLIDOS

df_candidatos_validos = df_candidatos_ies[
    ~df_candidatos_ies["NOMBRE_SOSPECHOSO"]
].copy()

df_candidatos_validos = (
    df_candidatos_validos
    .sort_values(
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "AÑO_ORIGEN",
            "FRECUENCIA",
        ],
        ascending=[
            True,
            False,
            False,
        ]
    )
)

dim_universidades = (
    df_candidatos_validos
    .drop_duplicates(
        subset="CÓDIGO DE LA INSTITUCIÓN",
        keep="first"
    )
    [
        [
            "CÓDIGO DE LA INSTITUCIÓN",
            "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
        ]
    ]
    .sort_values(
        "CÓDIGO DE LA INSTITUCIÓN"
    )
    .reset_index(drop=True)
)

dim_universidades.head(20) 



,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)
0,1101,UNIVERSIDAD NACIONAL DE COLOMBIA
1,1102,UNIVERSIDAD NACIONAL DE COLOMBIA
2,1103,UNIVERSIDAD NACIONAL DE COLOMBIA
3,1104,UNIVERSIDAD NACIONAL DE COLOMBIA
4,1105,UNIVERSIDAD PEDAGOGICA NACIONAL
5,1106,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
6,1107,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
7,1108,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
8,1109,UNIVERSIDAD PEDAGOGICA Y TECNOLOGICA DE COLOMB...
9,1110,UNIVERSIDAD DEL CAUCA


In [20]:
col_codigo = 'CÓDIGO DE LA INSTITUCIÓN'
col_ies = 'INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)'

print("Total de registros:", len(dim_universidades))

print("Códigos únicos:",
      dim_universidades[col_codigo].nunique())

print("Instituciones únicas:",
      dim_universidades[col_ies].nunique())

print("Instituciones con varios códigos:",
      (dim_universidades.groupby(col_ies)[col_codigo]
       .nunique() > 1).sum())



Total de registros: 336
Códigos únicos: 336
Instituciones únicas: 289
Instituciones con varios códigos: 21


In [21]:
# 3.7 EXPORTACIÓN DIM_UNIVERSIDADES

OUTPUT_DIM_UNIVERSIDADES = (
    PROCESSED / "04_Dim_Universidades.xlsx"
)

dim_universidades.to_excel(
    OUTPUT_DIM_UNIVERSIDADES,
    sheet_name="Dim_Universidades",
    index=False
)

print("Archivo generado correctamente:")
print(OUTPUT_DIM_UNIVERSIDADES)

Archivo generado correctamente:
C:\Users\fernanda.gonzalez\OneDrive - Grupo Planeta\DATA\Analytics Projects\Product_Analytics\data\processed\04_Dim_Universidades.xlsx


## 4. Construcción de la dimensión de programas

A partir del histórico SNIES consolidado se construye una dimensión única de programas académicos.

La dimensión utiliza `CÓDIGO SNIES DEL PROGRAMA` como llave y conserva una única denominación normalizada de `PROGRAMA ACADÉMICO` por código.

El proceso incluye la normalización del código SNIES, limpieza de las denominaciones académicas, identificación de variaciones históricas y selección del nombre válido más reciente.

Esta dimensión corresponde al catálogo de programas SNIES y no realiza todavía la homologación de programas equivalentes entre diferentes instituciones.

In [22]:
# 4.1 NORMALIZACIÓN DEL CÓDIGO SNIES DEL PROGRAMA

df_historico["CÓDIGO SNIES DEL PROGRAMA"] = normalizar_codigo(
    df_historico["CÓDIGO SNIES DEL PROGRAMA"]
)

# Validación
print(
    "Códigos SNIES únicos:",
    df_historico["CÓDIGO SNIES DEL PROGRAMA"].nunique()
)

print(
    "Códigos SNIES nulos:",
    df_historico["CÓDIGO SNIES DEL PROGRAMA"].isna().sum()
)

Códigos SNIES únicos: 15564
Códigos SNIES nulos: 0


In [23]:
# 4.2 BASE PARA DIMENSIÓN DE PROGRAMAS

df_programas_base = df_historico[
    [
        "CÓDIGO SNIES DEL PROGRAMA",
        "PROGRAMA ACADÉMICO",
        "AÑO_ORIGEN",
    ]
].copy()

df_programas_base = (
    df_programas_base[
        df_programas_base[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].notna()
    ]
    .copy()
)

df_programas_base[
    "PROGRAMA ACADÉMICO"
] = normalizar_texto(
    df_programas_base[
        "PROGRAMA ACADÉMICO"
    ]
)

df_programas_base = df_programas_base[
    df_programas_base[
        "PROGRAMA ACADÉMICO"
    ].notna()
].copy()

In [24]:
# 4.3 VALIDACIÓN DE VARIACIONES DE NOMBRE

variaciones_programa = (
    df_programas_base
    .groupby(
        "CÓDIGO SNIES DEL PROGRAMA"
    )["PROGRAMA ACADÉMICO"]
    .nunique()
    .sort_values(ascending=False)
)

print(
    "Códigos SNIES con más de un nombre:",
    (variaciones_programa > 1).sum()
)

variaciones_programa.head(20)

Códigos SNIES con más de un nombre: 301


CÓDIGO SNIES DEL PROGRAMA
19866     4
11452     3
106143    3
12503     3
12505     3
10711     3
19109     3
19641     3
2016      3
20886     3
4         3
4225      3
52395     3
52804     3
10110     3
100       2
10587     2
103772    2
102518    2
103049    2
Name: PROGRAMA ACADÉMICO, dtype: int64

In [25]:
# DETALLE DE PROGRAMAS CON VARIACIONES

codigos_programa_con_variaciones = variaciones_programa[
    variaciones_programa > 1
].index

df_variaciones_programa = (
    df_programas_base[
        df_programas_base[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].isin(codigos_programa_con_variaciones)
    ]
    [
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "PROGRAMA ACADÉMICO",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "PROGRAMA ACADÉMICO",
        ]
    )
    .reset_index(drop=True)
)

df_variaciones_programa.head(30)

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO
0,100,MAESTRIA EN INGENIERIA - GEOTECNIA
1,100,MAESTRÍA EN INGENIERÍA - GEOTECNIA
2,10016,ESPECIALIZACION EN RADIOLOGIA E IMAGENES
3,10016,ESPECIALIZACION EN RADIOLOGIA E IMAGENES DIAGN...
4,1008,ESPECIALIZACION EN GERENCIA DE CONSTRUCCIONES
5,1008,ESPECIALIZACIÓN EN GERENCIA DE CONSTRUCCIONES
6,10110,INGENIERA CIVIL
7,10110,INGENIER¿A CIVIL ARTICULADO POR CICLOS PROPED¿...
8,10110,INGENIERÍA CIVIL ARTICULADO POR CICLOS PROPEDÉ...
9,101353,ESPECIALIZACION EN ANESTESIOLOGIA CARDIOVASCULAR


In [26]:
# 4.4 VALIDACIÓN DE CARACTERES SOSPECHOSOS

df_programas_base["NOMBRE_SOSPECHOSO"] = (
    df_programas_base[
        "PROGRAMA ACADÉMICO"
    ]
    .str.contains(
        r"[¿�ÃÂ]",
        regex=True,
        na=False
    )
)

In [27]:
df_nombres_programa_sospechosos = (
    df_programas_base[
        df_programas_base["NOMBRE_SOSPECHOSO"]
    ]
    [
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "PROGRAMA ACADÉMICO",
            "AÑO_ORIGEN",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "AÑO_ORIGEN",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Nombres de programas con caracteres sospechosos:",
    len(df_nombres_programa_sospechosos)
)

df_nombres_programa_sospechosos.head(20)

Nombres de programas con caracteres sospechosos: 194


,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,AÑO_ORIGEN
0,10110,INGENIER¿A CIVIL ARTICULADO POR CICLOS PROPED¿...,2022
1,10110,INGENIER¿A CIVIL ARTICULADO POR CICLOS PROPED¿...,2023
2,10110,INGENIER¿A CIVIL ARTICULADO POR CICLOS PROPED¿...,2025
3,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,2020
4,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,2021
5,101698,COMUNICACIÓN SOCIAL ¿ PERIODISMO,2020
6,101698,COMUNICACIÓN SOCIAL ¿ PERIODISMO,2021
7,101698,COMUNICACIÓN SOCIAL ¿ PERIODISMO,2022
8,101698,COMUNICACIÓN SOCIAL ¿ PERIODISMO,2023
9,101698,COMUNICACIÓN SOCIAL ¿ PERIODISMO,2025


In [28]:
# 4.4.1 CLAVE NORMALIZADA PARA COMPARAR PROGRAMAS
def crear_clave_programa(serie):
    """
    Crea una versión auxiliar del nombre para comparar variantes.

    No reemplaza PROGRAMA ACADÉMICO.
    """

    return (
        serie
        .astype("string")
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.upper()
        .str.replace(r"[^A-Z0-9 ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


df_programas_base["PROGRAMA_KEY"] = crear_clave_programa(
    df_programas_base["PROGRAMA ACADÉMICO"]
)

In [29]:
# 4.4.2 CLASIFICACIÓN DE VARIACIONES

diagnostico_programas = (
    df_programas_base
    .groupby("CÓDIGO SNIES DEL PROGRAMA")
    .agg(
        NOMBRES_OBSERVADOS=(
            "PROGRAMA ACADÉMICO",
            "nunique"
        ),
        NOMBRES_NORMALIZADOS=(
            "PROGRAMA_KEY",
            "nunique"
        ),
        NOMBRES_SOSPECHOSOS=(
            "NOMBRE_SOSPECHOSO",
            "sum"
        ),
    )
    .sort_values(
        [
            "NOMBRES_NORMALIZADOS",
            "NOMBRES_OBSERVADOS",
        ],
        ascending=False
    )
)

diagnostico_programas.head(30)

,NOMBRES_OBSERVADOS,NOMBRES_NORMALIZADOS,NOMBRES_SOSPECHOSOS
CÓDIGO SNIES DEL PROGRAMA,,,
10110,3,3,12
20886,3,3,8
19866,4,2,2
106143,3,2,0
10711,3,2,2
11452,3,2,27
12503,3,2,6
12505,3,2,8
19109,3,2,8


In [30]:
print(
    "Códigos SNIES afectados:",
    df_nombres_programa_sospechosos[
        "CÓDIGO SNIES DEL PROGRAMA"
    ].nunique()
)

Códigos SNIES afectados: 105


In [31]:
print(
    "Nombres sospechosos distintos:",
    df_nombres_programa_sospechosos[
        "PROGRAMA ACADÉMICO"
    ].nunique()
)

Nombres sospechosos distintos: 84


In [32]:
# 4.4.2 CLASIFICACIÓN DE VARIACIONES

df_programas_base["PROGRAMA_SOSPECHOSO"] = (
    df_programas_base["PROGRAMA ACADÉMICO"]
    .where(df_programas_base["NOMBRE_SOSPECHOSO"])
)

df_programas_base["PROGRAMA_VALIDO"] = (
    df_programas_base["PROGRAMA ACADÉMICO"]
    .where(~df_programas_base["NOMBRE_SOSPECHOSO"])
)

df_programas_base["PROGRAMA_KEY_VALIDO"] = (
    df_programas_base["PROGRAMA_KEY"]
    .where(~df_programas_base["NOMBRE_SOSPECHOSO"])
)

NOMBRES_OBSERVADOS
- nombres literalmente diferentes encontrados

NOMBRES_NORMALIZADOS
- variantes después de quitar tildes/puntuación/etc.

NOMBRES_VALIDOS
- nombres diferentes que NO tienen caracteres dañados

NOMBRES_NORMALIZADOS_VALIDOS
- diferencias reales entre nombres limpios

NOMBRES_SOSPECHOSOS
- nombres dañados distintos

REGISTROS_SOSPECHOSOS
- cantidad de filas históricas afectadas

In [33]:
diagnostico_programas = (
    df_programas_base
    .groupby("CÓDIGO SNIES DEL PROGRAMA")
    .agg(
        NOMBRES_OBSERVADOS=(
            "PROGRAMA ACADÉMICO",
            "nunique"
        ),
        NOMBRES_NORMALIZADOS=(
            "PROGRAMA_KEY",
            "nunique"
        ),
        NOMBRES_VALIDOS=(
            "PROGRAMA_VALIDO",
            "nunique"
        ),
        NOMBRES_NORMALIZADOS_VALIDOS=(
            "PROGRAMA_KEY_VALIDO",
            "nunique"
        ),
        NOMBRES_SOSPECHOSOS=(
            "PROGRAMA_SOSPECHOSO",
            "nunique"
        ),
        REGISTROS_SOSPECHOSOS=(
            "NOMBRE_SOSPECHOSO",
            "sum"
        ),
    )
    .sort_values(
        [
            "NOMBRES_NORMALIZADOS_VALIDOS",
            "NOMBRES_OBSERVADOS",
        ],
        ascending=False
    )
)

diagnostico_programas.head(30)

,NOMBRES_OBSERVADOS,NOMBRES_NORMALIZADOS,NOMBRES_VALIDOS,NOMBRES_NORMALIZADOS_VALIDOS,NOMBRES_SOSPECHOSOS,REGISTROS_SOSPECHOSOS
CÓDIGO SNIES DEL PROGRAMA,,,,,,
10110,3,3,2,2,1,12
106143,3,2,3,2,0,0
19109,3,2,2,2,1,8
2016,3,2,2,2,1,4
20886,3,3,2,2,1,8
10016,2,2,2,2,0,0
101362,2,2,2,2,0,0
102604,2,2,2,2,0,0
102790,2,2,2,2,0,0


In [34]:
# ============================================================
# VALIDACIÓN DE COBERTURA DE NOMBRES VÁLIDOS
# ============================================================

codigos_totales = set(
    df_programas_base[
        "CÓDIGO SNIES DEL PROGRAMA"
    ].dropna()
)

codigos_con_nombre_valido = set(
    df_programas_base.loc[
        ~df_programas_base["NOMBRE_SOSPECHOSO"],
        "CÓDIGO SNIES DEL PROGRAMA"
    ].dropna()
)

codigos_sin_nombre_valido = sorted(
    codigos_totales - codigos_con_nombre_valido
)

print(
    "Códigos SNIES totales:",
    len(codigos_totales)
)

print(
    "Códigos sin ninguna denominación válida:",
    len(codigos_sin_nombre_valido)
)

codigos_sin_nombre_valido[:20]

Códigos SNIES totales: 15564
Códigos sin ninguna denominación válida: 14


['101414',
 '102058',
 '102319',
 '102478',
 '102554',
 '103648',
 '103707',
 '105161',
 '111728',
 '14295',
 '15955',
 '2134',
 '5274',
 '91157']

### 4.4.3 Revisión manual de códigos sin denominación válida

Se identifican los códigos SNIES para los cuales todas las denominaciones observadas presentan problemas de codificación.

Estos casos se revisan manualmente utilizando la institución, el año, el nivel de formación y la modalidad como contexto, evitando inferir automáticamente caracteres perdidos.

In [35]:
# 4.4.3 REVISIÓN MANUAL DE CÓDIGOS SIN NOMBRE VÁLIDO

columnas_revision = [
    "CÓDIGO SNIES DEL PROGRAMA",
    "PROGRAMA ACADÉMICO",
    "AÑO_ORIGEN",
    "CÓDIGO DE LA INSTITUCIÓN",
    "INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES)",
    "NIVEL DE FORMACIÓN",
    "MODALIDAD",
]

# Solo usar columnas que realmente existan en df_historico
columnas_revision = [
    columna
    for columna in columnas_revision
    if columna in df_historico.columns
]

df_revision_manual_programas = (
    df_historico[
        df_historico[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].isin(codigos_sin_nombre_valido)
    ][columnas_revision]
    .drop_duplicates()
    .sort_values(
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "AÑO_ORIGEN",
        ]
    )
    .reset_index(drop=True)
)

df_revision_manual_programas

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,AÑO_ORIGEN,CÓDIGO DE LA INSTITUCIÓN,INSTITUCIÓN DE EDUCACIÓN SUPERIOR (IES),NIVEL DE FORMACIÓN,MODALIDAD
0,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,2020,2745,FUNDACI¿N UNIVERSITARIA COMPENSAR,Tecnológica,Distancia (virtual)
1,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,2021,2745,FUNDACIÓN UNIVERSITARIA COMPENSAR,Tecnológico,Virtual
2,102058,TECNOLOGIA EN GESTI¿N DE EXPORTACIONES E IMPOR...,2020,2836,CORPORACION UNIVERSITARIA EMPRESARIAL DE SALAM...,Tecnológica,Presencial
3,102319,MAESTR¿A EN CALIDAD EN SERVICIOS DE SALUD,2020,1804,UNIVERSIDAD AUTONOMA DEL CARIBE- UNIAUTONOMA,Maestría,Distancia (virtual)
4,102478,MAESTRIA EN SALUD P¿BLICA,2020,2728,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Maestría,Presencial
5,102478,MAESTRIA EN SALUD P¿BLICA,2021,2728,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Maestría,Presencial
6,102554,LICENCIATURA EN PEDAGOG¿A INFANTIL,2020,2728,FUNDACION UNIVERSITARIA DEL AREA ANDINA,Universitaria,Distancia (virtual)
7,103648,ADMINISTRACI¿N DE LA SALUD OCUPACIONAL,2020,9905,FUNDACION ESCUELA TECNOLOGICA DE NEIVA - JESUS...,Universitaria,Presencial
8,103648,ADMINISTRACI¿N DE LA SALUD OCUPACIONAL,2021,9905,FUNDACION ESCUELA TECNOLOGICA DE NEIVA - JESUS...,Universitario,Presencial
9,103707,TECNOLOGIA EN GESTI¿N DE SISTEMAS MECATRONICOS,2020,3706,FUNDACION CENTRO COLOMBIANO DE ESTUDIOS PROFES...,Tecnológica,Presencial


In [36]:
# 4.4.4 CORRECCIONES MANUALES DE PROGRAMAS

CORRECCIONES_MANUALES_PROGRAMAS = {
    "101414": "TECNOLOGÍA EN GESTIÓN DE PROCESOS ADMINISTRATIVOS",
    "102058": "TECNOLOGÍA EN GESTIÓN DE EXPORTACIONES E IMPORTACIONES",
    "102319": "MAESTRÍA EN CALIDAD EN SERVICIOS DE SALUD",
    "102478": "MAESTRÍA EN SALUD PÚBLICA",
    "102554": "LICENCIATURA EN PEDAGOGÍA INFANTIL",
    "103648": "ADMINISTRACIÓN DE LA SALUD OCUPACIONAL",
    "103707": "TECNOLOGÍA EN GESTIÓN DE SISTEMAS MECATRÓNICOS",
    "105161": "ESPECIALIZACIÓN EN INNOVACIÓN EDUCATIVA DIGITAL",
    "111728": "TÉCNICA PROFESIONAL EN PROCESOS GRÁFICOS",
    "14295": "ESPECIALIZACIÓN EN VIOLENCIA INTRAFAMILIAR: NIÑEZ Y ADOLESCENCIA",
    "15955": "TECNOLOGÍA EN SISTEMAS DE INFORMACIÓN",
    "2134": "TECNOLOGÍA EN PRODUCCIÓN INDUSTRIAL",
    "5274": "TECNOLOGÍA EN GESTIÓN CONTABLE Y FINANCIERA",
    "91157": "TECNOLOGÍA EN CONTABILIDAD Y FINANZAS",
}

df_programas_base.loc[
    df_programas_base["CÓDIGO SNIES DEL PROGRAMA"] == "102058",
    "PROGRAMA ACADÉMICO"
].drop_duplicates().tolist()

df_programas_base["PROGRAMA_ACADÉMICO_CANÓNICO"] = (
    df_programas_base[
        "CÓDIGO SNIES DEL PROGRAMA"
    ]
    .map(CORRECCIONES_MANUALES_PROGRAMAS)
    .fillna(
        df_programas_base["PROGRAMA ACADÉMICO"]
    )
)

df_programas_base["PROGRAMA_KEY"] = crear_clave_programa(
    df_programas_base[
        "PROGRAMA_ACADÉMICO_CANÓNICO"
    ]
)

df_programas_base["NOMBRE_SOSPECHOSO"] = (
    df_programas_base[
        "PROGRAMA_ACADÉMICO_CANÓNICO"
    ]
    .str.contains(
        r"[¿�ÃÂ]",
        regex=True,
        na=False
    )
)

print(
    "Registros sospechosos después de corrección:",
    df_programas_base["NOMBRE_SOSPECHOSO"].sum()
)

Registros sospechosos después de corrección: 751


In [37]:
validacion_correcciones = (
    df_programas_base[
        df_programas_base[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].isin(CORRECCIONES_MANUALES_PROGRAMAS)
    ]
    [
        [
            "CÓDIGO SNIES DEL PROGRAMA",
            "PROGRAMA ACADÉMICO",
            "PROGRAMA_ACADÉMICO_CANÓNICO",
        ]
    ]
    .drop_duplicates()
    .sort_values("CÓDIGO SNIES DEL PROGRAMA")
    .reset_index(drop=True)
)

validacion_correcciones

,CÓDIGO SNIES DEL PROGRAMA,PROGRAMA ACADÉMICO,PROGRAMA_ACADÉMICO_CANÓNICO
0,101414,TECNOLOG¿A EN GESTI¿N DE PROCESOS ADMINISTRATIVOS,TECNOLOGÍA EN GESTIÓN DE PROCESOS ADMINISTRATIVOS
1,102058,TECNOLOGIA EN GESTI¿N DE EXPORTACIONES E IMPOR...,TECNOLOGÍA EN GESTIÓN DE EXPORTACIONES E IMPOR...
2,102319,MAESTR¿A EN CALIDAD EN SERVICIOS DE SALUD,MAESTRÍA EN CALIDAD EN SERVICIOS DE SALUD
3,102478,MAESTRIA EN SALUD P¿BLICA,MAESTRÍA EN SALUD PÚBLICA
4,102554,LICENCIATURA EN PEDAGOG¿A INFANTIL,LICENCIATURA EN PEDAGOGÍA INFANTIL
5,103648,ADMINISTRACI¿N DE LA SALUD OCUPACIONAL,ADMINISTRACIÓN DE LA SALUD OCUPACIONAL
6,103707,TECNOLOGIA EN GESTI¿N DE SISTEMAS MECATRONICOS,TECNOLOGÍA EN GESTIÓN DE SISTEMAS MECATRÓNICOS
7,105161,ESPECIALIZACI¿N EN INNOVACI¿N EDUCATIVA DIGITAL,ESPECIALIZACIÓN EN INNOVACIÓN EDUCATIVA DIGITAL
8,111728,TECNIC¿ PROFESIONAL EN PROCESOS GR¿FICOS,TÉCNICA PROFESIONAL EN PROCESOS GRÁFICOS
9,14295,ESPECIALIZACION EN VIOLENCIA INTRAFAMILIAR:NI¿...,ESPECIALIZACIÓN EN VIOLENCIA INTRAFAMILIAR: NI...


In [38]:
codigos_sin_nombre_valido = sorted(
    set(
        df_programas_base[
            "CÓDIGO SNIES DEL PROGRAMA"
        ].dropna()
    )
    -
    set(
        df_programas_base.loc[
            ~df_programas_base["NOMBRE_SOSPECHOSO"],
            "CÓDIGO SNIES DEL PROGRAMA"
        ].dropna()
    )
)

print(
    "Códigos sin denominación válida después de corrección:",
    len(codigos_sin_nombre_valido)
)

Códigos sin denominación válida después de corrección: 0


## 3. Lectura eficiente del `.xlsx`

El archivo contiene seis hojas y más de 300 mil registros. Para mantener bajo el consumo de memoria, se recorren las filas directamente desde los XML internos del libro. Esta capa solo se encarga de lectura; toda la lógica de negocio sigue estando en las funciones de normalización anteriores.

In [ ]:

NS = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
REL = '{http://schemas.openxmlformats.org/officeDocument/2006/relationships}'
ROW_RE = re.compile(rb'<row\b[^>]*>(.*?)</row>', re.S)
CELL_RE = re.compile(rb'<c\b([^>]*)>(.*?)</c>', re.S)
V_RE = re.compile(rb'<v>(.*?)</v>', re.S)
T_RE = re.compile(rb'<t(?:\s[^>]*)?>(.*?)</t>', re.S)
R_RE = re.compile(rb'\br="([A-Z]+)\d+"')
TYPE_RE = re.compile(rb'\bt="([^"]+)"')


def col_idx(letras_bytes):
    n = 0
    for ch in letras_bytes:
        n = n * 26 + ch - 64
    return n - 1


def parse_cells(row_bytes, shared_strings):
    """Devuelve {indice_columna: valor} para una fila del XML de Excel."""
    out = {}
    for cm in CELL_RE.finditer(row_bytes):
        attrs, inner = cm.group(1), cm.group(2)
        rm = R_RE.search(attrs)
        if not rm:
            continue
        idx = col_idx(rm.group(1))
        tm = TYPE_RE.search(attrs)
        typ = tm.group(1) if tm else b''
        vm = V_RE.search(inner)
        if vm:
            raw = vm.group(1).decode('utf-8', 'replace')
            value = shared_strings[int(raw)] if typ == b's' else raw
        else:
            value = ''.join(html.unescape(t.decode('utf-8', 'replace')) for t in T_RE.findall(inner))
        out[idx] = value
    return out


def metadata_libro(z):
    ssroot = ET.fromstring(z.read('xl/sharedStrings.xml'))
    shared = [''.join((t.text or '') for t in si.iter(NS + 't')) for si in ssroot.findall(NS + 'si')]

    wbroot = ET.fromstring(z.read('xl/workbook.xml'))
    relroot = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
    rels = {r.attrib['Id']: r.attrib['Target'] for r in relroot}
    sheets = [
        (s.attrib['name'], 'xl/' + rels[s.attrib[REL + 'id']].lstrip('/'))
        for s in wbroot.find(NS + 'sheets')
    ]
    return shared, sheets

with zipfile.ZipFile(INPUT) as z:
    shared_strings, hojas = metadata_libro(z)

print('Hojas detectadas:', [h[0] for h in hojas])


## 4. Procesamiento completo: normalización + agregados para EDA y dimensiones

Cada registro se normaliza antes de alimentar cualquier salida. No se utilizan los nombres crudos para construir las dimensiones.

Se acumulan únicamente los conjuntos y contadores necesarios, evitando duplicar las 310 mil filas en memoria. Si necesitas conservar el histórico normalizado completo, activa `EXPORTAR_HISTORICO_NORMALIZADO=True`.

In [ ]:

rows_year = Counter()
ies_year = defaultdict(set)
prog_year = defaultdict(set)
mat_year = Counter()
nulls = defaultdict(Counter)

ies_names = defaultdict(Counter)
ies_attrs = defaultdict(lambda: defaultdict(Counter))
program_variants = {}                    # key -> nombre display
program_variant_ies = defaultdict(set)
program_variant_snies = defaultdict(set)
program_variant_example_ies = {}
manual_pairs = set()
ibero_programs = {}                       # key -> nombre canónico Ibero
ibero_rows = set()

KEY_QUALITY = [
    'CODIGO_IES','IES','CODIGO_SNIES_PROGRAMA','PROGRAMA_ACADEMICO',
    'NIVEL_FORMACION','MODALIDAD','MATRICULADOS_PRIMER_CURSO'
]

if EXPORTAR_HISTORICO_NORMALIZADO:
    gz_file = gzip.open(PROCESSED / 'Historico_SNIES_normalizado.csv.gz', 'wt', encoding='utf-8-sig', newline='')
    normalized_writer = csv.DictWriter(gz_file, fieldnames=CANON)
    normalized_writer.writeheader()
else:
    gz_file = None
    normalized_writer = None

with zipfile.ZipFile(INPUT) as z:
    shared_strings, hojas = metadata_libro(z)

    for sheet_name, target in hojas:
        data = z.read(target)
        row_iter = ROW_RE.finditer(data)

        # Cabecera de cada año -> columna canónica
        header_cells = parse_cells(next(row_iter).group(1), shared_strings)
        header_map = {i: ALIASES.get(clave_ascii(v)) for i, v in header_cells.items()}
        header_map = {i: c for i, c in header_map.items() if c}

        n_sheet = 0
        for rm in row_iter:
            n_sheet += 1
            cells = parse_cells(rm.group(1), shared_strings)

            # Normalización de TODAS las columnas mapeadas de la fila
            d = {c: '' for c in CANON}
            for i, c in header_map.items():
                value = cells.get(i, '')
                d[c] = normalizar_texto_categoria(c, value) if c in TEXT_COLS else normalizar_codigo(value, c)

            # Si el año viene anómalo, la hoja es la referencia de respaldo.
            if d['ANIO'] not in {'2020','2021','2022','2023','2024','2025'}:
                d['ANIO'] = sheet_name

            if normalized_writer:
                normalized_writer.writerow(d)

            year = d['ANIO']
            code_ies = d['CODIGO_IES']
            snies = d['CODIGO_SNIES_PROGRAMA']
            programa = d['PROGRAMA_ACADEMICO']
            pkey = clave_ascii(programa)

            rows_year[year] += 1
            if code_ies: ies_year[year].add(code_ies)
            if snies: prog_year[year].add(snies)
            try: mat_year[year] += int(d['MATRICULADOS_PRIMER_CURSO'] or 0)
            except ValueError: pass

            for c in KEY_QUALITY:
                if not d[c]: nulls[year][c] += 1

            if code_ies:
                if d['IES']:
                    ies_names[code_ies][d['IES']] += 1
                for c in ['IES_PADRE','TIPO_IES','ID_SECTOR_IES','SECTOR_IES','ID_CARACTER_IES',
                          'CARACTER_IES','COD_DEP_IES','DEP_IES','COD_MUN_IES','MUN_IES']:
                    if d[c]: ies_attrs[code_ies][c][d[c]] += 1

            if pkey:
                program_variants.setdefault(pkey, programa)
                if code_ies:
                    program_variant_ies[pkey].add(code_ies)
                    manual_pairs.add((code_ies, pkey))
                if snies:
                    program_variant_snies[pkey].add(snies)
                if pkey not in program_variant_example_ies and d['IES']:
                    program_variant_example_ies[pkey] = d['IES']

                if code_ies == CODIGO_IBERO:
                    ibero_programs.setdefault(pkey, programa)
                    ibero_rows.add((snies, pkey, d['NIVEL_FORMACION'], d['MODALIDAD']))

        print(f'{sheet_name}: {n_sheet:,} filas procesadas')

if gz_file:
    gz_file.close()

print('\nTotal filas:', f'{sum(rows_year.values()):,}')
print('IES únicas:', len(ies_names))
print('Nombres de programa únicos:', len(program_variants))
print('Programas canónicos Ibero:', len(ibero_programs))


## 5. EDA

El EDA se concentra en volumen, cobertura y calidad de los campos esenciales para la homologación. No se altera el histórico; solo se generan métricas de control.

In [ ]:

eda_resumen = [
    {
        'ANIO': y,
        'FILAS': rows_year[y],
        'IES_UNICAS': len(ies_year[y]),
        'CODIGOS_SNIES_PROGRAMA_UNICOS': len(prog_year[y]),
        'MATRICULADOS_PRIMER_CURSO_TOTAL': mat_year[y]
    }
    for y in sorted(rows_year)
]

eda_calidad = [
    {
        'ANIO': y,
        'COLUMNA': c,
        'NULOS': nulls[y][c],
        'PORC_NULOS': nulls[y][c] / rows_year[y] if rows_year[y] else 0
    }
    for y in sorted(rows_year)
    for c in KEY_QUALITY
]

for r in eda_resumen:
    print(r)

# Visual simple: evolución del número de registros por año
plt.figure(figsize=(8,4))
plt.bar([r['ANIO'] for r in eda_resumen], [r['FILAS'] for r in eda_resumen])
plt.title('Registros SNIES por año')
plt.xlabel('Año')
plt.ylabel('Filas')
plt.show()


## 6. Base Ibero 2830

Esta es la referencia canónica inicial. El **nombre de programa de Ibero** se toma únicamente de los registros del código IES 2830 ya normalizados. Se conserva código SNIES, nivel de formación y modalidad para no perder trazabilidad.

In [ ]:

ibero_base = [
    {
        'CODIGO_IES': CODIGO_IBERO,
        'IES': 'Corporación universitaria iberoamericana',
        'CODIGO_SNIES_PROGRAMA': snies,
        'PROGRAMA_IBERO': ibero_programs[pkey],
        'PROGRAMA_IBERO_KEY': pkey,
        'NIVEL_FORMACION': nivel,
        'MODALIDAD': modalidad
    }
    for snies, pkey, nivel, modalidad in sorted(ibero_rows, key=lambda x: (x[1], x[0], x[2], x[3]))
]

print('Filas base Ibero:', len(ibero_base))
print('Nombres canónicos Ibero:', len(ibero_programs))


## 7. Primera dimensión de homologación automática

El grano principal es **un nombre normalizado de programa SNIES**. Cada nombre se compara contra los 53 nombres canónicos de Ibero.

El score combina:
- **40% contención de tokens**: útil cuando el nombre externo amplía el nombre Ibero.
- **25% Jaccard**: mide solapamiento de palabras relevantes.
- **20% similitud por conjunto de tokens**.
- **15% similitud secuencial**.

Umbrales iniciales: `ALTA ≥ 0.82`, `MEDIA ≥ 0.68`, `BAJA ≥ 0.56`; por debajo se deja `SIN_MATCH`. Son umbrales de trabajo, no reglas definitivas.

In [ ]:

STOP = {'DE','DEL','LA','LAS','EL','LOS','Y','E','EN','PARA','CON','POR','AL'}

def tokens_relevantes(s):
    return {x for x in s.split() if x not in STOP and len(x) > 1}


def similarity_score(a, b):
    if a == b:
        return 1.0
    ta, tb = tokens_relevantes(a), tokens_relevantes(b)

    if RAPIDFUZZ:
        seq = fuzz.ratio(a, b) / 100.0
        token = fuzz.token_set_ratio(a, b) / 100.0
    else:
        seq = SequenceMatcher(None, a, b).ratio()
        token = seq

    if not ta or not tb:
        return seq

    inter = len(ta & tb)
    containment = inter / min(len(ta), len(tb))
    jaccard = inter / len(ta | tb)
    return min(1.0, 0.40*containment + 0.25*jaccard + 0.20*token + 0.15*seq)

ibero_keys = list(ibero_programs)
dim_programa_auto = []

for pkey, pdisp in sorted(program_variants.items(), key=lambda kv: kv[1]):
    best_key, best_score = max(
        ((ibk, similarity_score(pkey, ibk)) for ibk in ibero_keys),
        key=lambda x: x[1]
    )

    if best_score >= 0.82: confianza = 'ALTA'
    elif best_score >= 0.68: confianza = 'MEDIA'
    elif best_score >= 0.56: confianza = 'BAJA'
    else: confianza = 'SIN_MATCH'

    dim_programa_auto.append({
        'CODIGO_IES_REFERENCIA': CODIGO_IBERO,
        'PROGRAMA_IBERO': ibero_programs.get(best_key, '') if confianza != 'SIN_MATCH' else '',
        'PROGRAMA_NOMBRE': pdisp,
        'PROGRAMA_NOMBRE_KEY': pkey,
        'SCORE_MATCH': round(best_score, 4),
        'CONFIANZA': confianza,
        'N_IES_QUE_USAN_NOMBRE': len(program_variant_ies[pkey]),
        'N_CODIGOS_SNIES_ASOCIADOS': len(program_variant_snies[pkey]),
        'IES_EJEMPLO': program_variant_example_ies.get(pkey, '')
    })

print(Counter(r['CONFIANZA'] for r in dim_programa_auto))


## 8. Base para homologación manual

Esta segunda base tiene un grano distinto: **una fila por combinación `CODIGO_IES + PROGRAMA_NORMALIZADO`**. La columna `PROGRAMA_IBERO_MANUAL` queda vacía para que puedas asignar manualmente el programa Ibero y luego contrastar el resultado con la homologación automática.

In [ ]:

programas_manual = []
for codigo_ies, pkey in sorted(manual_pairs, key=lambda x: (int(x[0]) if x[0].isdigit() else 999999999, x[1])):
    ies = ies_names[codigo_ies].most_common(1)[0][0] if ies_names[codigo_ies] else ''
    programas_manual.append({
        'CODIGO_IES': codigo_ies,
        'IES': ies,
        'PROGRAMA_NORMALIZADO': program_variants[pkey],
        'PROGRAMA_KEY': pkey,
        'PROGRAMA_IBERO_MANUAL': ''
    })

print('Pares IES-programa para revisión manual:', len(programas_manual))


## 9. Dimensión de universidades

Se crea una fila por código IES. Cuando un atributo cambia de representación entre años, se conserva el valor **más frecuente** del histórico para estabilizar la dimensión y se añade `IES_KEY` para comparaciones.

In [ ]:

dim_universidades = []
ATTRS_IES = ['IES_PADRE','TIPO_IES','ID_SECTOR_IES','SECTOR_IES','ID_CARACTER_IES','CARACTER_IES',
             'COD_DEP_IES','DEP_IES','COD_MUN_IES','MUN_IES']

for codigo_ies in sorted(ies_names, key=lambda x: int(x) if x.isdigit() else 999999999):
    row = {
        'CODIGO_IES': codigo_ies,
        'IES': ies_names[codigo_ies].most_common(1)[0][0] if ies_names[codigo_ies] else ''
    }
    for c in ATTRS_IES:
        row[c] = ies_attrs[codigo_ies][c].most_common(1)[0][0] if ies_attrs[codigo_ies][c] else ''
    row['IES_KEY'] = clave_ascii(row['IES'])
    dim_universidades.append(row)

print('IES en dimensión:', len(dim_universidades))


## 10. Exportación a Excel

Se exportan archivos separados porque cumplen propósitos distintos: referencia Ibero, homologación automática, validación manual, dimensión IES y EDA. El formato incluye encabezados, filtros y congelación de la primera fila.

In [ ]:

HEADER_FILL = PatternFill('solid', fgColor='163A5F')
HEADER_FONT = Font(color='FFFFFF', bold=True)
THIN = Side(style='thin', color='D9E2F3')


def exportar_excel(rows, path, sheet_name, confidence=False):
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name[:31]

    if not rows:
        wb.save(path)
        return

    headers = list(rows[0].keys())
    ws.append(headers)
    for r in rows:
        ws.append([r.get(h, '') for h in headers])

    for c in ws[1]:
        c.fill = HEADER_FILL
        c.font = HEADER_FONT
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for i, h in enumerate(headers, 1):
        width = 14
        if any(k in h for k in ['PROGRAMA','IES','NOMBRE','KEY']): width = 42
        ws.column_dimensions[get_column_letter(i)].width = width

    if confidence and 'CONFIANZA' in headers:
        cidx = headers.index('CONFIANZA') + 1
        for cell in ws[get_column_letter(cidx)][1:]:
            if cell.value == 'ALTA': cell.fill = PatternFill('solid', fgColor='C6EFCE')
            elif cell.value == 'MEDIA': cell.fill = PatternFill('solid', fgColor='FFEB9C')
            elif cell.value == 'BAJA': cell.fill = PatternFill('solid', fgColor='FCE4D6')
            elif cell.value == 'SIN_MATCH': cell.fill = PatternFill('solid', fgColor='F4CCCC')

    wb.save(path)


def exportar_eda(resumen, calidad, path):
    wb = Workbook()
    ws = wb.active; ws.title = 'Resumen_anual'
    for sheet, rows in [(ws, resumen), (wb.create_sheet('Calidad_campos_clave'), calidad)]:
        headers = list(rows[0].keys())
        sheet.append(headers)
        for r in rows: sheet.append([r[h] for h in headers])
        for c in sheet[1]:
            c.fill = HEADER_FILL; c.font = HEADER_FONT; c.alignment = Alignment(horizontal='center', wrap_text=True)
        sheet.freeze_panes = 'A2'; sheet.auto_filter.ref = sheet.dimensions
        for i,h in enumerate(headers,1): sheet.column_dimensions[get_column_letter(i)].width = 28 if h=='COLUMNA' else 22
        if 'PORC_NULOS' in headers:
            col = get_column_letter(headers.index('PORC_NULOS') + 1)
            for cell in sheet[col][1:]: cell.number_format = '0.00%'
    wb.save(path)

exportar_excel(ibero_base, PROCESSED/'01_Ibero_2830_programas.xlsx', 'Ibero_2830')
exportar_excel(dim_programa_auto, PROCESSED/'02_Dim_Programa_Agrupacion_Automatica.xlsx', 'Dim_Programa_Auto', confidence=True)
exportar_excel(programas_manual, PROCESSED/'03_Programas_Normalizados_Manual.xlsx', 'Programas_Manual')
exportar_excel(dim_universidades, PROCESSED/'04_Dim_Universidades.xlsx', 'Dim_Universidades')
exportar_eda(eda_resumen, eda_calidad, PROCESSED/'05_EDA_SNIES.xlsx')

print('Archivos generados:')
for p in sorted(PROCESSED.glob('*.xlsx')):
    print('-', p.name)


## 11. Criterio de uso recomendado

- **Confianza ALTA:** candidata a aceptación automática, pero conviene revisar una muestra antes de productivizar.
- **MEDIA:** revisión manual obligatoria en esta primera iteración.
- **BAJA:** tratar como sugerencia, no como homologación final.
- **SIN_MATCH:** no asignar automáticamente.

La validación manual (`03_...`) permite construir un **golden set**. En una segunda iteración se compara `PROGRAMA_IBERO_MANUAL` contra `PROGRAMA_IBERO` y se recalibran pesos/umbrales. Solo después de estabilizar esta dimensión recomendaría incorporar CECOS al modelo.